In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

BIDS_DIR = Path('/data/ds-valuecapture')
subjects = [1, 2]
sessions = [1, 2]

In [ ]:
dfs = []

for sub in subjects:
    for ses in sessions:
        func_dir = BIDS_DIR / f'sub-{sub:02d}' / f'ses-{ses}' / 'func'
        for tsv in sorted(func_dir.glob(f'sub-{sub:02d}_ses-{ses}_task-valuecapture_run-*_events.tsv')):
            run = int(tsv.stem.split('run-')[1].split('_')[0])
            df = pd.read_csv(tsv, sep='\t')
            df['subject'] = sub
            df['session'] = ses
            df['run'] = run
            dfs.append(df)

df = pd.concat(dfs, ignore_index=True)
print(f'Loaded {len(dfs)} runs, {df.subject.nunique()} subjects')

In [ ]:
# One row per trial: iti1 phase has rt and correct filled in
trials = df[df['event_type'] == 'iti1'].copy()

def condition(row):
    if not row['distractor_present']:
        return 'absent'
    return f'rank-{int(row["value_rank"])}'

trials['condition'] = trials.apply(condition, axis=1)
trials['rt_ms'] = trials['rt'] * 1000

print(trials['condition'].value_counts())

In [ ]:
CONDITION_ORDER = ['absent', 'rank-0', 'rank-1', 'rank-2']

grp = ['subject', 'session', 'condition']
mean_rt  = trials[trials['correct'] == True].groupby(grp)['rt_ms'].mean().rename('mean_rt')
accuracy = trials.groupby(grp)['correct'].mean().rename('accuracy')

summary = pd.concat([mean_rt, accuracy], axis=1).reset_index()
summary['condition'] = pd.Categorical(summary['condition'], categories=CONDITION_ORDER, ordered=True)
summary = summary.sort_values(['subject', 'session', 'condition'])
summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

sns.barplot(data=summary, x='condition', y='mean_rt', hue='subject',
            order=CONDITION_ORDER, ax=axes[0])
axes[0].set_title('Mean RT (correct trials)')
axes[0].set_ylabel('RT (ms)')
axes[0].set_xlabel('Condition')

sns.barplot(data=summary, x='condition', y='accuracy', hue='subject',
            order=CONDITION_ORDER, ax=axes[1])
axes[1].set_title('Accuracy')
axes[1].set_ylabel('Proportion correct')
axes[1].set_xlabel('Condition')
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.show()

In [ ]:
# Grand mean across sessions
grand = summary.groupby(['subject', 'condition'])[['mean_rt', 'accuracy']].mean().round(1)
grand